# Inteligência Artifical e Aprendizado de Máquina — Entrega 1
## Agente de Pendências — baseline

**Objetivo:** receber uma solicitação do aluno, reconhecer seu assunto com uma Árvore de Decisão simples e usar regras para apresentar checklist, prioridade inicial, próxima ação, justificativa e setor responsável. A decisão final é sempre do atendente.

Este notebook reúne o código, o relatório técnico e a primeira versão do Model Card.

## 1. Análise dos dados

O projeto recebeu `Contato.xlsx`, `Financeiro.xlsx`, `Historico.xlsx`, `Matriculas.xlsx` e `Relacionamentos.xlsx`. Utilizamos a `Base Unificada.csv` já disponibilizada para consultar indicadores por aluno. O notebook mostra a quantidade de registros, alunos e colunas e a presença de valores vazios.

| Fonte | Conteúdo utilizado ou relacionado | Observação |
|---|---|---|
| Contato | Informações cadastrais | Fonte original recebida |
| Financeiro | Parcelas, atrasos e acordos | Pode conter vários registros por aluno |
| Histórico | Informações acadêmicas | Pode conter vários registros por aluno |
| Matrículas | Informações de matrícula | Fonte original recebida |
| Relacionamentos | Histórico de contatos | Contato não confirma pendência atual |
| Base Unificada.csv | Indicadores consolidados | Base usada no protótipo |

**Granularidade:** a base unificada é consultada por `ID_ALUNO`. As fontes com vários eventos por estudante precisam ser agregadas antes de uma junção por esse ID.

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# No Colab, faça upload de Base Unificada.csv quando solicitado.
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass

df = pd.read_csv("Base Unificada.csv")
print("Linhas:", len(df), "| Colunas:", len(df.columns))
print("Alunos distintos:", df["ID_ALUNO"].nunique())
display(df.head())

## 2. Preparação dos dados

Verificamos campos vazios e IDs duplicados. Para a consulta demonstrativa, mantemos uma linha por aluno. Valores financeiros inválidos são tratados como ausentes, **não como ausência de pendência**. Não usamos nomes, sexo ou idade para atribuir prioridade.

In [ ]:
print("IDs vazios:", df["ID_ALUNO"].isna().sum())
print("IDs duplicados:", df["ID_ALUNO"].duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("Valores vazios"))

dados = df.dropna(subset=["ID_ALUNO"]).drop_duplicates("ID_ALUNO").copy()
colunas_numericas = ["pct_parcelas_em_aberto", "pct_parcelas_acordo", "atraso_medio_dias", "total_contatos", "media_assiduidade"]
for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
print("Registros após preparação:", len(dados))
display(dados[["ID_ALUNO"] + colunas_numericas].head())

## 3. Seleção de atributos

| Dimensão | Atributo | Uso |
|---|---|---|
| Identificação | `ID_ALUNO` | Localizar registro (não entra no modelo) |
| Financeira | `pct_parcelas_em_aberto` | Indicar necessidade de conferir parcelas |
| Financeira | `atraso_medio_dias` | Apoiar priorização inicial |
| Financeira | `pct_parcelas_acordo` | Indicar acordo a verificar |
| Bolsas | `tem_bolsa` | Verificar registro de bolsa, não sua renovação |
| Acadêmica | `media_assiduidade` | Contexto, não prova pendência |
| Atendimento | `total_contatos` | Contexto de contatos anteriores |
| Solicitação | Texto digitado | Entrada da Árvore de Decisão para identificar o assunto |

O modelo de ML usa **somente o texto da solicitação**. As demais colunas são consultadas pelas regras do agente. Não usamos `evadiu`, pois o objetivo não é prever evasão.

## 4. Métricas planejadas

Nesta entrega, medimos a **acurácia de classificação do assunto** em um pequeno conjunto de frases demonstrativas e executamos testes das regras de resposta. Os exemplos de treinamento são criados apenas para esta demonstração: a acurácia **não mede o desempenho com alunos reais**. Na próxima entrega, com solicitações reais revisadas, avaliar acurácia por categoria, encaminhamentos corretos e pendências confirmadas.

## 5b. Baseline do Agente de Pendências

### 5b.1 Modelo inicial: Árvore de Decisão

Criamos frases de exemplo para quatro assuntos e treinamos uma Árvore de Decisão pequena. O `CountVectorizer` transforma palavras em números para o modelo. **Os exemplos abaixo são fictícios e servem apenas para mostrar um modelo treinado funcionando.**

In [ ]:
exemplos = {
    "financeiro": [
        "tenho boleto atrasado", "quero verificar parcelas em aberto", "preciso pagar mensalidade",
        "tenho dívida de pagamento", "como consultar meu acordo financeiro", "meu boleto está vencido",
        "quero regularizar parcelas", "minha mensalidade está atrasada",
        "preciso consultar pagamento", "há algum débito financeiro"],
    "rematricula": [
        "quero fazer rematrícula", "como renovar minha matrícula", "preciso me rematricular",
        "qual o procedimento de rematrícula", "quero continuar matriculado", "como fazer matrícula do semestre",
        "tenho dúvidas sobre rematrícula", "preciso renovar matrícula",
        "quando posso fazer rematrícula", "como concluir a matrícula"],
    "bolsa": [
        "quero renovar minha bolsa", "tenho dúvida sobre bolsa de estudos", "como solicitar bolsa",
        "minha bolsa está ativa", "preciso consultar desconto da bolsa", "quero saber sobre auxílio estudantil",
        "minha bolsa precisa de renovação", "como verificar benefício estudantil",
        "preciso regularizar bolsa", "quero informação sobre bolsa"],
    "documentos": [
        "quais documentos faltam", "preciso entregar documentação", "tenho documento pendente",
        "como enviar comprovante", "falta algum documento meu", "quero verificar documentação",
        "preciso atualizar meus documentos", "entrega de histórico escolar",
        "meu comprovante foi recebido", "quero consultar documento"],
}
frases = [frase for lista in exemplos.values() for frase in lista]
assuntos = [assunto for assunto, lista in exemplos.items() for _ in lista]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    frases, assuntos, test_size=0.25, random_state=42, stratify=assuntos
)
modelo = make_pipeline(CountVectorizer(), DecisionTreeClassifier(max_depth=6, random_state=42))
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)
print("Acurácia demonstrativa:", round(accuracy_score(y_teste, previsoes) * 100, 1), "%")
print(classification_report(y_teste, previsoes, zero_division=0))
display(pd.DataFrame({"Pedido de teste": X_teste, "Esperado": y_teste, "Previsto": previsoes}).head())

### 5b.2 Regras e tabela de decisão

O modelo identifica o assunto; as regras consultam os dados e indicam o que o atendente deve verificar. **As prioridades são provisórias**, não regras oficiais do ASA. Um indicador financeiro não comprova bloqueio de rematrícula.

In [ ]:
tabela_decisao = pd.DataFrame([
    ["Financeiro: parcelas em aberto e atraso >= 30 dias", "Conferir parcelas e acordos", "Alta (provisória)", "Financeiro"],
    ["Financeiro: parcelas em aberto e atraso < 30 dias", "Conferir parcelas", "Média (provisória)", "Financeiro"],
    ["Financeiro: sem indicador suficiente", "Consultar situação atual", "A verificar", "Financeiro"],
    ["Rematrícula", "Conferir calendário, requisitos e eventuais bloqueios", "A verificar", "Secretaria"],
    ["Bolsa", "Conferir regras, situação e renovação", "A verificar", "Bolsas/atendimento"],
    ["Documentos", "Conferir lista exigida e recebimento", "A verificar", "Secretaria"],
], columns=["Condição", "Resultado", "Prioridade", "Setor"])
display(tabela_decisao)

In [ ]:
def orientar_aluno(id_aluno, pedido):
    aluno = dados.loc[dados["ID_ALUNO"].astype(str) == str(id_aluno)]
    if aluno.empty:
        return {"Assunto": "Não localizado", "Checklist": "Conferir identificação",
                "Prioridade": "A verificar", "Próxima ação": "Localizar cadastro",
                "Justificativa": "ID não encontrado", "Setor": "Atendimento"}

    assunto = modelo.predict([pedido])[0]
    registro = aluno.iloc[0]
    if assunto == "financeiro":
        aberto = registro["pct_parcelas_em_aberto"]
        atraso = registro["atraso_medio_dias"]
        if pd.isna(aberto):
            checklist, prioridade, motivo = "Consultar parcelas no sistema", "A verificar", "Indicador ausente"
        elif aberto > 0:
            checklist = "Conferir parcelas em aberto e eventuais acordos"
            prioridade = "Alta (provisória)" if pd.notna(atraso) and atraso >= 30 else "Média (provisória)"
            motivo = "Indicador de parcelas em aberto; atraso médio: " + (str(round(atraso, 1)) if pd.notna(atraso) else "não informado")
        else:
            checklist, prioridade, motivo = "Confirmar situação financeira atual", "A verificar", "Sem parcelas em aberto no indicador resumido"
        return {"Assunto": assunto, "Checklist": checklist, "Prioridade": prioridade,
                "Próxima ação": "Conferir no sistema financeiro e orientar o aluno",
                "Justificativa": motivo, "Setor": "Financeiro"}
    orientacoes = {
        "rematricula": ("Conferir calendário, documentos e requisitos da rematrícula", "Consultar situação atual e regras oficiais", "Secretaria acadêmica"),
        "bolsa": ("Conferir registro, condições e documentos da bolsa", "Consultar situação e regras de renovação", "Setor de bolsas"),
        "documentos": ("Conferir documentos exigidos e recebimento", "Verificar documentos pendentes no sistema", "Secretaria acadêmica"),
    }
    checklist, proxima, setor = orientacoes[assunto]
    return {"Assunto": assunto, "Checklist": checklist, "Prioridade": "A verificar",
            "Próxima ação": proxima, "Justificativa": "A base não confirma a pendência específica deste procedimento",
            "Setor": setor}

# Demonstração com ID fictício retirado da base;
id_exemplo = dados.iloc[0]["ID_ALUNO"]
for pedido in ["quero verificar parcelas em aberto", "quero fazer rematrícula",
               "quero renovar minha bolsa", "quais documentos faltam"]:
    print("\nSOLICITAÇÃO:", pedido)
    display(pd.DataFrame([orientar_aluno(id_exemplo, pedido)]))

### 5b.3 Verificação inicial

A tabela de classificação mostra se o modelo reconhece os assuntos nos exemplos separados para teste. As quatro solicitações acima demonstram a resposta completa do agente. Antes de uso real, precisamos validar frases, regras e prioridades com atendentes do ASA.

## 6. Limitações conhecidas

- O classificador aprendeu com **40 frases fictícias**; pode errar com perguntas reais ou misturadas.
- A base unificada traz indicadores resumidos, mas não confirma documentos faltantes, solicitações abertas ou prazos.
- Um pedido pode envolver mais de um assunto; esta versão retorna apenas uma categoria.
- A acurácia demonstrativa não comprova eficácia operacional.

## 7. Próximos passos

Na Entrega 2: reunir solicitações reais autorizadas e rotuladas, incluir regras oficiais para rematrícula, bolsas e documentos, avaliar os encaminhamentos com atendentes e melhorar a identificação de pedidos que envolvem mais de um assunto.

---
# Parte 2 — Model Card (primeira versão)

**Nome:** ASA-Connect 
**Versão:** 1.0 — baseline da Entrega 1  
**Algoritmo:** `CountVectorizer` + `DecisionTreeClassifier` para identificar o assunto; `if/else` e tabela de decisão para orientar o atendimento.  

**Uso pretendido:** apoiar atendentes na triagem inicial de solicitações sobre financeiro, rematrícula, bolsas e documentos. Produz checklist, prioridade provisória, próxima ação, justificativa e setor. Não aprova/nega pedidos, não aplica sanções e não substitui análise humana.

**Dados de treinamento:** 40 frases **fictícias** criadas para demonstração, distribuídas igualmente entre quatro assuntos. A base unificada fornecida pelo projeto é consultada pelas regras; **não foi usada para treinar o classificador de textos**. Seu período de cobertura não foi confirmado.

**Avaliação:** a célula de treinamento apresenta acurácia e relatório de classificação em 25% das frases demonstrativas, separados do treino. Os resultados são ilustrativos e não equivalem à avaliação com solicitações reais. As quatro respostas do agente mostram o funcionamento inicial.

**Considerações éticas:** consultar apenas dados autorizados, não publicar identificadores de alunos, verificar informações em sistemas oficiais e manter um atendente responsável por decisões.

**Limitações:** poucos exemplos fictícios, ausência de regras institucionais completas e de status documental e prazos atualizados.

**Histórico:** versão 1.0 — baseline inicial da Entrega 1.